# BiRD reactor transport refactor for cell-engine coupling

_Investigation `bird-transport` — coder reproduction notebook._

**Question.** Can the BiRD 0D reactor physics be refactored so that biomass is an
input — an external cell engine owns growth and substrate exchange while
the reactor process owns only gas–liquid transport — and does the
resulting coupled system exercise the reactor→cell feedback that a
single-process reactor cannot?

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-bioreactordesign/viva-bioreactordesign').is_dir():
    REPO = Path('/home/runner/work/viva-bioreactordesign/viva-bioreactordesign')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_bioreactordesign.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: `bird-01-transport-process`

**Question.** Can the gas–liquid transport physics be extracted into a shared module
and exposed as a biomass-as-input BiRDTransportProcess that emits only
the transport contribution, while the standalone BiRDReactorProcess (now
transport + an extracted MonodCellProcess) keeps its pre-refactor
behavior?

**Objective.** Extract the transport physics into a shared module, factor the internal
Monod term into a MonodCellProcess, and build BiRDTransportProcess; then
measure emitted deltas, contract conformance, and the standalone reactor's
trajectories to determine that responsibilities are cleanly split and
standalone behavior is preserved.

**Hypothesis.** A shared transport module consumed by both processes lets
BiRDTransportProcess read biomass/substrate/dissolved-gas as inputs and
emit only kLa·(C*−C) deltas (zero for glucose); the biomass/consumption
term factors into a MonodCellProcess that conforms to the cell-side
contract; standalone BiRDReactorProcess (transport + MonodCellProcess) is
unchanged within tolerance and the bird_disable_internal_biomass flag is
removed as obsolete.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `standalone-reactor` | `viva_bioreactordesign.composites.reactor` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_bioreactordesign.composites.reactor`** — `spec_viva_bioreactordesign_composites_reactor` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_bioreactordesign_composites_reactor = load_spec(REPO / 'viva_bioreactordesign/composites/reactor.composite.yaml')
describe_spec(spec_viva_bioreactordesign_composites_reactor)

In [ ]:
# === Edit parameters for composite 'reactor' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'reactor'  (local:BiRDReactorProcess)
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['interval'] = 1.0
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['reactor_type'] = 'bubble_column'
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['volume_L'] = 20.0
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['diameter_m'] = 0.2
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['liquid_height_m'] = 0.5
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['gas_flow_rate_Lpm'] = 1.0
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['temperature_K'] = 298.15
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['pressure_atm'] = 1.0
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['o2_fraction_inlet'] = 0.21
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['co2_fraction_inlet'] = 0.0004
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['mean_bubble_diameter_mm'] = 3.0
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['initial_biomass_gL'] = 0.5
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['initial_do_mgL'] = 8.0
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['initial_dco2_mgL'] = 0.5
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['max_growth_rate_per_h'] = 0.4
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['ks_oxygen_mgL'] = 0.2
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['yield_biomass_o2'] = 1.2
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['maintenance_coeff_per_h'] = 0.01
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['respiratory_quotient'] = 1.0
spec_viva_bioreactordesign_composites_reactor['state']['reactor']['config']['impeller_power_W'] = 0.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bird-01-transport-process ===
STUDY = 'bird-01-transport-process'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

## Study: `bird-02-closed-loop-liveness`

**Question.** When BiRDTransportProcess is coupled to the Phase-1 MonodCellProcess at a
static high biomass density, does the reactor→cell feedback fire —
dissolved O2 dropping below saturation as consumption outpaces transport —
with O2 mass balance closing across cell and reactor contributions?

**Objective.** Compose BiRDTransportProcess with MonodCellProcess at a constant high
biomass density and measure dissolved O2 and the O2 balance to determine
that the closed loop is live, not merely wired.

**Hypothesis.** At an impactful biomass density, steady-state dissolved O2 settles below
saturation, d[O2]/dt equals reactor transport minus cell consumption at
every step, and removing the biomass input returns O2 to saturation —
confirming a live, bidirectional loop.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `coupled-reactor-cell` | `viva_bioreactordesign.composites.coupled_reactor_cell` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_bioreactordesign.composites.coupled_reactor_cell`** — `spec_viva_bioreactordesign_composites_coupled_reactor_cell` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_bioreactordesign_composites_coupled_reactor_cell = load_spec(REPO / 'viva_bioreactordesign/composites/coupled_reactor_cell.composite.yaml')
describe_spec(spec_viva_bioreactordesign_composites_coupled_reactor_cell)

In [ ]:
# === Edit parameters for composite 'coupled_reactor_cell' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'transport'  (local:BiRDTransportProcess)
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['interval'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['reactor_type'] = 'bubble_column'
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['volume_L'] = 20.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['diameter_m'] = 0.2
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['gas_flow_rate_Lpm'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['temperature_K'] = 298.15
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['pressure_atm'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['o2_fraction_inlet'] = 0.21
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['co2_fraction_inlet'] = 0.0004
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['mean_bubble_diameter_mm'] = 3.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['impeller_power_W'] = 0.0

# process 'cell'  (local:MonodCellProcess)
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['interval'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['initial_biomass_gL'] = 5.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['max_growth_rate_per_h'] = 0.4
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['ks_oxygen_mgL'] = 0.2
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['yield_biomass_o2'] = 1.2
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['maintenance_coeff_per_h'] = 0.01
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['respiratory_quotient'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['growth_enabled'] = False

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bird-02-closed-loop-liveness ===
STUDY = 'bird-02-closed-loop-liveness'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

## Study: `bird-03-coupling-interval`

**Question.** Can the reactor coupling/update interval be exposed as a tunable parameter,
and does the coupled trajectory converge as the interval decreases?

**Objective.** Expose the coupling interval as a parameter and measure the dissolved-O2
trajectory across interval values to determine convergence behavior.

**Hypothesis.** Making the interval configurable and shrinking it produces a convergent
dissolved-O2 trajectory (changes below tolerance under interval halving),
so the interval can trade fidelity against cost without changing the
qualitative result.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `coupled-reactor-cell` | `viva_bioreactordesign.composites.coupled_reactor_cell` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_bioreactordesign.composites.coupled_reactor_cell`** — `spec_viva_bioreactordesign_composites_coupled_reactor_cell` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_bioreactordesign_composites_coupled_reactor_cell = load_spec(REPO / 'viva_bioreactordesign/composites/coupled_reactor_cell.composite.yaml')
describe_spec(spec_viva_bioreactordesign_composites_coupled_reactor_cell)

In [ ]:
# === Edit parameters for composite 'coupled_reactor_cell' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'transport'  (local:BiRDTransportProcess)
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['interval'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['reactor_type'] = 'bubble_column'
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['volume_L'] = 20.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['diameter_m'] = 0.2
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['gas_flow_rate_Lpm'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['temperature_K'] = 298.15
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['pressure_atm'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['o2_fraction_inlet'] = 0.21
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['co2_fraction_inlet'] = 0.0004
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['mean_bubble_diameter_mm'] = 3.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['impeller_power_W'] = 0.0

# process 'cell'  (local:MonodCellProcess)
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['interval'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['initial_biomass_gL'] = 5.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['max_growth_rate_per_h'] = 0.4
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['ks_oxygen_mgL'] = 0.2
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['yield_biomass_o2'] = 1.2
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['maintenance_coeff_per_h'] = 0.01
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['respiratory_quotient'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['growth_enabled'] = False

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bird-03-coupling-interval ===
STUDY = 'bird-03-coupling-interval'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

## Study: `bird-04-reactor-geometry`

**Question.** Can the reactor support a stirred-tank kLa correlation (the benchmark
geometry) alongside the bubble-column form, selectable by configuration?

**Objective.** Add a selectable stirred-tank kLa correlation and measure kLa against the
published correlation to determine the model can be parameterized to a
specific vessel.

**Hypothesis.** A geometry configuration field selects the kLa correlation, and under
stirred-tank geometry kLa matches the published correlation for the
configured power input and superficial gas velocity within tolerance.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `coupled-reactor-cell` | `viva_bioreactordesign.composites.coupled_reactor_cell` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_bioreactordesign.composites.coupled_reactor_cell`** — `spec_viva_bioreactordesign_composites_coupled_reactor_cell` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_bioreactordesign_composites_coupled_reactor_cell = load_spec(REPO / 'viva_bioreactordesign/composites/coupled_reactor_cell.composite.yaml')
describe_spec(spec_viva_bioreactordesign_composites_coupled_reactor_cell)

In [ ]:
# === Edit parameters for composite 'coupled_reactor_cell' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'transport'  (local:BiRDTransportProcess)
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['interval'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['reactor_type'] = 'bubble_column'
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['volume_L'] = 20.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['diameter_m'] = 0.2
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['gas_flow_rate_Lpm'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['temperature_K'] = 298.15
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['pressure_atm'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['o2_fraction_inlet'] = 0.21
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['co2_fraction_inlet'] = 0.0004
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['mean_bubble_diameter_mm'] = 3.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['transport']['config']['impeller_power_W'] = 0.0

# process 'cell'  (local:MonodCellProcess)
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['interval'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['initial_biomass_gL'] = 5.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['max_growth_rate_per_h'] = 0.4
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['ks_oxygen_mgL'] = 0.2
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['yield_biomass_o2'] = 1.2
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['maintenance_coeff_per_h'] = 0.01
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['respiratory_quotient'] = 1.0
spec_viva_bioreactordesign_composites_coupled_reactor_cell['state']['cell']['config']['growth_enabled'] = False

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bird-04-reactor-geometry ===
STUDY = 'bird-04-reactor-geometry'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")